<a href="https://colab.research.google.com/github/jaewon00o0/kbu-2-1-/blob/main/%EB%8D%B0%EC%9D%B4%ED%84%B0%EA%B4%80%EB%A6%AC%EB%A1%A0%204%EC%9B%94%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

1. 데이터 로드 및 샘플링

In [2]:
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

In [3]:
data, labels = [], []
for i, category in enumerate(categories):
    # 필수 요구 사항에 맞춰 각 주제별 20개 샘플 추출 (이미지상 100을 20으로 수정)
    indices = np.where(newsgroups.target == i)[0][:20]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

CountVectorizer 설정 및 변환

In [4]:
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

3. 분류 함수 구현

In [10]:
def classify_text(input_text):
    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)
    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]

4.테스트 실행

In [11]:
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god."
]

for s in test_sentences:
    cat, score = classify_text(s)
    print(f"문장: {s[:30]:<30}... | 예측: {cat:<20} | 유사도: {score:.4f}")

문장: The rocket launched into orbit... | 예측: sci.space            | 유사도: 0.0870
문장: A new 3D rendering technique f... | 예측: comp.graphics        | 유사도: 0.1952
문장: Theological discussions on fai... | 예측: talk.religion.misc   | 유사도: 0.3430


In [ ]:
Q1. CountVectorizer는 학습 데이터에 등장하는 단어들을 모아 고정된 크기의 단어 사전을 생성하고, 이를 기반으로 빈도수를 세어 벡터화합니다.
입력된 문장에서 불용어인 'the', 'with', 'a'가 제거되고 나면 'Exploring', 'mars', 'robotic', 'rover' 등의 단어가 남습니다.
그런데 학습 데이터로 각 카테고리당 단 20개의 적은 샘플만 사용했기 때문에, 이 핵심 단어들이 학습 데이터 내에 단 한 번도 등장하지 않았을 확률이 매우 높습니다. 이를 OOV 문제라고 합니다.
사전에 없는 단어들은 변환 과정에서 모두 무시되므로, 결과적으로 해당 입력 문장의 텍스트 벡터는 모든 값이 0이 됩니다. 영벡터와 다른
어떠한 문서 벡터를 비교하더라도 겹치는 단어가 없으므로 코사인 유사도는 0.0000으로 계산됩니다.

In [ ]:
Q2.
1.주제별 샘플 수를 20개에서 100개로 늘렸을 때의 변화:

변화 이유: 학습 데이터의 양이 5배 늘어나면서 CountVectorizer가 구축하는 단어 사전의 크기가 대폭 확장됩니다.

결과 예측: 'mars', 'orbit', 'god' 등 다양한 도메인 특화 단어들이 사전에 포함될 확률이 높아져 Q1에서 발생한 OOV 문제가 크게 완화되며, 유사도 점수와 분류 정확도가 상승합니다.

2.CountVectorizer 대신 TfidfVectorizer를 사용했을 때의 차이점:

변화 이유: 단순 빈도만 계산하면 문서의 길이나 흔한 단어에 의해 유사도가 왜곡될 수 있습니다. TF-IDF는 특정 문서에만 자주 등장하는 핵심 단어의 가중치를 높이고, 모든 문서에 뻔하게 등장하는 단어의 가중치는 낮춥니다.

결과 예측: 카테고리를 구분 짓는 변별력 있는 단어들의 영향력이 커지기 때문에, 단순히 단어가 많이 겹치는 것보다 주제적 연관성을 훨씬 더 정확하게 잡아내어 분류 성능이 개선됩니다.

3.ngram_range=(1, 2) 설정을 추가했을 때의 영향:
변화 이유: 기본 설정은 단어를 하나씩만 분리해서 보지만, Bigram(ngram_range=(1, 2)) 설정을 추가하면 인접한 두 단어의 묶음도 하나의 고유한 특징으로 취급합니다.

결과 예측: 단어의 문맥과 순서 정보가 어느 정도 벡터에 반영되므로, 특정 분야에서 자주 쓰이는 고유 명사나 숙어 형태를 더 잘 잡아내어 유사도 계산의 정교함이 높아집니다.

구현 가이드: 뉴스그룹 데이터 분류 시스템


In [12]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

1. 환경 설정 및 데이터 준비

In [13]:
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
print("데이터를 로드하는 중입니다...")

newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

data, labels = [], []
for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

print(f"총 추출된 문서 개수: {len(data)}개 (3개 주제 * 20개)")

데이터를 로드하는 중입니다...
총 추출된 문서 개수: 60개 (3개 주제 * 20개)


2. 텍스트 데이터의 수치화

In [14]:
vectorizer = CountVectorizer(stop_words='english')

count_matrix = vectorizer.fit_transform(data)

feature_names = vectorizer.get_feature_names_out()
print(f"생성된 어휘 사전 크기: {len(feature_names)}개 단어")
print(f"어휘 사전 샘플(앞 10개): {feature_names[:10]}\n")

생성된 어휘 사전 크기: 2342개 단어
어휘 사전 샘플(앞 10개): ['000' '02' '0865' '10' '100' '10bps' '11' '111' '12' '13']



3. 사용자 입력 및 유사도 분석 & 4. 결과 판정 및 출력

In [15]:
def classify_user_input(input_text):
    print(f"입력 문장: '{input_text}'")

    input_vec = vectorizer.transform([input_text])

    sim = cosine_similarity(input_vec, count_matrix)

    best_idx = np.argmax(sim)
    max_score = sim[0][best_idx]

    if max_score == 0.0:
         print(" 판정 결과: 학습 데이터에 입력 단어가 하나도 없습니다.")
    else:
        # 최종 매칭: 해당 인덱스의 라벨명(주제명)과 유사도 점수 출력
        best_label = labels[best_idx]
        print(f"=> 분류 예측: {best_label} | 최고 유사도: {max_score:.4f}")
    print("-" * 50)

In [16]:
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god.",
    "Exploring the mars with a robotic rover." # 유사도 0이 나올 확률이 높은 문장
]

In [17]:
print("=== 분류 테스트 결과 ===")
for s in test_sentences:
    classify_user_input(s)

=== 분류 테스트 결과 ===
입력 문장: 'The rocket launched into orbit.'
=> 분류 예측: sci.space | 최고 유사도: 0.0870
--------------------------------------------------
입력 문장: 'A new 3D rendering technique for graphics.'
=> 분류 예측: comp.graphics | 최고 유사도: 0.1952
--------------------------------------------------
입력 문장: 'Theological discussions on faith and god.'
=> 분류 예측: talk.religion.misc | 최고 유사도: 0.3430
--------------------------------------------------
입력 문장: 'Exploring the mars with a robotic rover.'
 판정 결과: 학습 데이터에 입력 단어가 하나도 없습니다.
--------------------------------------------------
